# Mapping Intertidal Elevation with Microsoft Planetary Computer and DEA Intertidal

### The "download-and-compute-locally" route — using the real DEA Intertidal algorithm, no Earth Engine account required

This notebook is the **second of two routes** in the intertidal-elevation tutorial. It
produces the same kind of product — an intertidal Digital Elevation Model (DEM) — but takes
a fundamentally different path from the Google Earth Engine (GEE) notebook:

| | **GEE route** (other notebook) | **This route — Planetary Computer + DEA** |
|---|---|---|
| Where computation happens | Google's cloud (server-side) | **Your machine** (download pixels, compute locally) |
| Account required | **Yes** — Google Cloud project + auth | **No** — STAC and Landsat/Sentinel-2 are anonymous |
| Elevation algorithm | A transparent re-implementation | **The real `intertidal.elevation()`** from DEA |
| Tide model | FES2022 (via `eo-tides`) | FES2022 (via `eo-tides`) — identical |
| Cost | Free tier, quota-limited | Your bandwidth + local compute |

#### When to use this route
Use it when you want the **production-grade DEA Intertidal algorithm** (Bishop-Taylor et al.,
2019) rather than a teaching re-implementation — it performs the smooth tidal-interval
interpolation and produces a per-pixel uncertainty layer — or when you simply cannot or do not
want to create a Google account. The trade-off is that you download the imagery, so very large
areas or long time series are limited by your local bandwidth and memory.

#### About the Planetary Computer account question
The Microsoft Planetary Computer STAC catalogue and its core open collections (Landsat,
Sentinel-2) are **anonymously accessible**. The function `planetary_computer.sign()` signs the
asset URLs so they can be read, and it works **without any key**. A free *subscription key*
only raises rate limits and is needed for certain protected/premium datasets (e.g. Sentinel-1
RTC). For the optical intertidal workflow here, **no key is required**.

> This notebook assumes the **`pc` environment** described in `README.md` (created with
> `uv sync --frozen` in the `pc/` folder). It is a *different* environment from the GEE
> notebook — the two are kept separate on purpose because their dependency trees conflict.


## Stage 0 — Verify the environment

Installation happens before Jupyter, from the terminal, using the locked `pc` environment:

```bash
cd pc
uv sync --frozen
uv run jupyter lab        # then open this notebook
```

This stage only *verifies*. Two things are worth knowing about why this environment is pinned
the way it is:

- **`pyTMD` is not installed directly** — `eo-tides` requires `pyTMD<3.0.0` and resolves the
  right version (2.2.9.1) itself.
- **`eodatasets3` is pinned to 1.9.3, not the latest.** `dea-intertidal` requires
  `rasterio<1.4.4`, but `eodatasets3==1.9.5` pulls in a newer `datacube` that forces
  `rasterio>=1.4.4` — a hard conflict. Version 1.9.3 allows an older `datacube` and resolves
  cleanly. This is exactly the kind of conflict a lock file is meant to pin down; `uv sync
  --frozen` reproduces the verified, working tree.


In [ ]:
import sys
assert sys.version_info >= (3, 12), (
    f"Python >= 3.12 required (you have {sys.version.split()[0]}). See README.md / pc environment."
)
print(f"Python {sys.version.split()[0]} — OK.")

import importlib
required = ["odc.stac", "planetary_computer", "pystac_client",
            "intertidal.elevation", "dea_tools", "eo_tides", "pyTMD",
            "rasterio", "geopandas", "rioxarray", "xarray", "numpy", "pandas"]
missing = []
for mod in required:
    try:
        importlib.import_module(mod)
    except ImportError as e:
        missing.append((mod, str(e)))
if missing:
    for m, e in missing:
        print("MISSING:", m, "->", e)
    raise SystemExit("Environment incomplete — see README.md (did you run `uv sync --frozen` in pc/?).")
print("All required packages import correctly, including intertidal.elevation.")

In [ ]:
# Resolved versions for your methods / supplementary information.
import importlib.metadata as im
for pkg in ["odc-stac", "planetary-computer", "pystac-client",
            "dea-intertidal", "dea-tools", "eodatasets3", "datacube",
            "eo-tides", "pyTMD", "rasterio", "geopandas", "rioxarray", "xarray"]:
    try:
        print(f"  {pkg:<20} {im.version(pkg)}")
    except im.PackageNotFoundError:
        print(f"  {pkg:<20} NOT INSTALLED")

## Stage 1 — The FES2022 tide model

This route uses the same tide model as the GEE notebook. If you have already set it up there,
point `TIDE_DIR` at the same folder.

A barotropic tide model such as FES2022 represents sea-surface height as a sum of harmonic
constituents (M2, S2, K1, …); evaluating that sum at a location and time reconstructs the
predicted tide height. Because the forcing is astronomical, the model extrapolates backwards in
time as well as forwards — relevant if you analyse historical Landsat-5 imagery.

Obtain FES2022 (free, with registration) from **AVISO+**
(https://www.aviso.altimetry.fr/en/data/data-access/registration-form.html), requesting the
**FES tide** product, and arrange the `fes2022b/ocean_tide/*.nc` constituent files as below.
This manual step is unavoidable: AVISO requires authenticated download.

```
TIDE_DIR/
└── fes2022b/
    └── ocean_tide/
        ├── m2_fes2022.nc
        ├── s2_fes2022.nc
        └── ...
```


In [ ]:
import os
from pathlib import Path

TIDE_DIR = "./tide_models"      # <-- EDIT: same folder as in the GEE notebook, if you have it
TIDE_MODEL = "FES2022"
os.environ["EO_TIDES_TIDE_MODELS"] = TIDE_DIR

ocean_tide = Path(TIDE_DIR) / "fes2022b" / "ocean_tide"
if ocean_tide.exists():
    ncs = sorted(ocean_tide.glob("*.nc"))
    print(f"Found {len(ncs)} constituent files in {ocean_tide}")
    if len(ncs) < 30:
        print("Note: a complete FES2022 set has ~34 constituents; a partial set degrades accuracy.")
else:
    print("Not found:", ocean_tide, "\nDownload FES2022 (see text) and set TIDE_DIR.")

## Stage 2 — Study area and observation window

As in the GEE route, define a bounding box (WGS84) and a time window. Because this route
**downloads** imagery, keep the area modest while learning — every scene is transferred to your
machine. A few kilometres across and a 2–4 year window is a sensible starting point.


In [ ]:
# === EDIT to your study area ===
WEST, SOUTH, EAST, NORTH = 3.96, 51.55, 4.10, 51.62   # example: part of the Oosterschelde, NL
SITE_NAME = "study_site"

bbox = (WEST, SOUTH, EAST, NORTH)
mid_lon, mid_lat = (WEST + EAST) / 2, (SOUTH + NORTH) / 2

START, END = "2021-01-01", "2024-12-31"
MAX_CLOUD = 80          # %, scene-level pre-filter

print(f"Site {SITE_NAME}: {bbox}")
print(f"Window {START} to {END}, cloud ceiling {MAX_CLOUD}%")

## Stage 3 — Querying Planetary Computer (STAC)

We search the Planetary Computer **STAC** (SpatioTemporal Asset Catalog) for Landsat Collection
2 Level 2 scenes over the AOI and window, then sign the results so their assets are readable.
The signing step (`planetary_computer.sign_inplace`) is what replaces an account — it attaches
short-lived access tokens to the asset URLs, anonymously.

This mirrors the `pc_load` helper from the original research pipeline.


In [ ]:
import pystac_client
import planetary_computer
import odc.stac

PC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"

catalog = pystac_client.Client.open(PC_URL, modifier=planetary_computer.sign_inplace)

search = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=bbox,
    datetime=f"{START}/{END}",
    query={
        "eo:cloud_cover": {"lt": MAX_CLOUD},
        "platform": {"in": ["landsat-5", "landsat-7", "landsat-8", "landsat-9"]},
        "landsat:collection_category": {"in": ["T1"]},
    },
)
items = search.item_collection()
print(f"Found {len(items)} Landsat scenes.")
if len(items) == 0:
    raise SystemExit("No scenes — widen the window, the AOI, or the cloud ceiling.")
if len(items) < 20:
    print("Note: few scenes -> the elevation fit will be less reliable.")

## Stage 4 — Loading the imagery as an xarray dataset

`odc-stac` lazily loads the signed STAC items into an `xarray` dataset on a common grid. We
request the bands needed for NDWI (green, NIR) plus the QA band for cloud masking, reproject to
a local UTM grid (so distances are in metres), and resample to 30 m (Landsat native).

`chunks={}` keeps the load lazy (backed by Dask) so nothing is read until needed.


In [ ]:
ds = odc.stac.load(
    items,
    bands=["green", "nir08", "qa_pixel"],
    bbox=bbox,
    crs="utm",
    resolution=30,
    chunks={"x": 2048, "y": 2048},
    groupby="solar_day",
    resampling={"qa_pixel": "nearest", "*": "cubic"},
    fail_on_error=False,
)
print("Loaded dataset dimensions:", dict(ds.sizes))
print("Variables:", list(ds.data_vars))

## Stage 5 — Cloud masking, scaling and NDWI

We apply the same physics as the GEE route, here on the local arrays:

1. **Cloud/shadow mask** from `qa_pixel` (bit 3 = cloud, bit 4 = cloud shadow).
2. **Scale** Collection 2 L2 digital numbers to surface reflectance: `DN x 0.0000275 - 0.2`.
3. **NDWI** = (green − NIR) / (green + NIR), the wet/dry discriminator (McFeeters, 1996).

The DEA `elevation()` function expects an NDWI variable, so that is what we hand it.


In [ ]:
import numpy as np

# Cloud + shadow mask
cloud = (np.bitwise_and(ds.qa_pixel, 1 << 3) | np.bitwise_and(ds.qa_pixel, 1 << 4)) == 0
ds = ds.where(cloud).drop_vars("qa_pixel")

# Scale to surface reflectance, clip to physical range
ds = (ds.where(ds != 0) * 0.0000275 - 0.2).clip(0, 1)

# NDWI
ndwi = ((ds.green - ds.nir08) / (ds.green + ds.nir08)).rename("ndwi")
ndwi_ds = ndwi.to_dataset()
print("NDWI computed. Dataset ready for elevation().")

## Stage 6 — Running the real DEA Intertidal `elevation()`

This is the heart of this route. `intertidal.elevation()` performs the production DEA Intertidal
workflow: it tide-tags every observation using `eo-tides`/FES2022, models the per-pixel
relationship between observed wet/dry state and tide height, and returns an elevation surface
**plus a per-pixel uncertainty layer** — the key advantage over the teaching re-implementation
in the GEE notebook.

> **This step computes locally and can take a while** — it loads the masked NDWI stack into
> memory and fits every pixel. For a small AOI this is minutes; scale the AOI up cautiously.

The function returns the elevation dataset (the exact returned object can be a tuple depending
on version; we handle both).


In [ ]:
from intertidal.elevation import elevation

result = elevation(ndwi_ds, tide_model=TIDE_MODEL, tide_model_dir=TIDE_DIR)
ds_elev = result[0] if isinstance(result, tuple) else result

print("elevation() returned variables:", list(ds_elev.data_vars))
# Typically includes: elevation, elevation_uncertainty, qa_ndwi_freq, qa_ndwi_corr, qa_count_clear

## Stage 7 — Inspecting elevation and its uncertainty

We plot the elevation surface and, alongside it, the uncertainty layer. The uncertainty is what
makes a DEA-style product defensible: it tells you *where* the estimate is trustworthy. Expect
higher uncertainty near the edges of the sampled tidal range, where few observations constrain
the wet/dry transition.


In [ ]:
import matplotlib.pyplot as plt

has_unc = "elevation_uncertainty" in ds_elev
fig, axes = plt.subplots(1, 2 if has_unc else 1, figsize=(13 if has_unc else 7, 6), squeeze=False)

ds_elev["elevation"].plot.imshow(ax=axes[0, 0], cmap="viridis",
                                 cbar_kwargs={"label": "Elevation (m, MSL)", "shrink": 0.7})
axes[0, 0].set_title(f"{SITE_NAME} — intertidal elevation (DEA)")
axes[0, 0].set_aspect("equal"); axes[0, 0].set_xticks([]); axes[0, 0].set_yticks([])

if has_unc:
    ds_elev["elevation_uncertainty"].plot.imshow(ax=axes[0, 1], cmap="magma",
                                cbar_kwargs={"label": "Uncertainty (m)", "shrink": 0.7})
    axes[0, 1].set_title("Per-pixel uncertainty")
    axes[0, 1].set_aspect("equal"); axes[0, 1].set_xticks([]); axes[0, 1].set_yticks([])

fig.tight_layout()
plt.show()

## Stage 8 — Saving the result

We save the elevation dataset (and its QA layers) to NetCDF, and the elevation band to a
GeoTIFF for use in GIS. Because the data are already local arrays, this is a direct write — no
cloud export step.


In [ ]:
out_nc = f"{SITE_NAME}_elevation_PC.nc"
ds_elev.to_netcdf(out_nc)
print("Wrote", out_nc)

# GeoTIFF of the elevation band (requires rioxarray; CRS is carried from the UTM load)
try:
    ds_elev["elevation"].rio.to_raster(f"{SITE_NAME}_elevation_PC.tif")
    print("Wrote", f"{SITE_NAME}_elevation_PC.tif")
except Exception as e:
    print("GeoTIFF export skipped:", e)

## Discussion — how this route compares

You have now produced an intertidal DEM with the **production DEA Intertidal algorithm**, tide-
tagged with FES2022, without any cloud account.

**What you gain over the GEE route**
- The real interval-interpolation fit and a **per-pixel uncertainty** layer.
- No account/auth; full local control over the data and intermediate products.
- The exact method used in the original research pipeline, so results are directly comparable.

**What it costs**
- You download every scene: bandwidth- and memory-bound, so large AOIs/long series are harder.
- A heavier, more conflict-prone dependency tree (hence the pinned, locked `pc` environment).

**Shared caveats** (identical to the GEE route)
- *Tidal sampling bias*: Landsat's fixed overpass time under-samples some tidal phases; elevations
  near the extremes of the *sampled* range are poorly constrained.
- *Datum*: heights are relative to model MSL; apply an offset for NAP/LAT and state it explicitly.
- *Morphological stationarity*: the window must be short enough that the bed is approximately stable.
- *Validation*: compare against an independent DEM (e.g. lidar DTM); report bias, MAE, RMSE.

#### Choosing between the two routes
For teaching the *mechanism*, the GEE notebook is clearer and needs no downloads. For a
*defensible product* with uncertainty, this PC/DEA route is the one to cite. Both share the same
tide model and the same physical assumptions, so they are two implementations of one method —
not two different methods.

#### Key references
- McFeeters, S.K. (1996). NDWI. *Int. J. Remote Sensing*.
- Bishop-Taylor, R. et al. (2019). Between the tides: modelling Australia's exposed intertidal zone. *Estuarine, Coastal and Shelf Science*.
- Lyard, F.H. et al. (2021). FES2014/FES2022 global ocean tide atlas. *Ocean Science*.
